# 第 5 章 · Tool Use（Vertex AI Search）

书里这版用的是 **`agents.VSearchAgent`——这个类在 ADK 2.13 里已经不存在了**，
`from google.adk import agents` 里没有任何 VSearch 相关名字。现在的等价物是一个**工具**：
`VertexAiSearchTool(data_store_id=...)`，挂到普通 `LlmAgent` 的 `tools=[]` 上。

也就是说书里「搜索 agent 是特殊类」的写法，已经统一成「搜索只是又一个工具」。

⚠️ **本机跑不了**：需要 Vertex AI 项目 + 真实的 datastore（不是 API key 能解决的），
而且 `DiscoveryEngineSearchTool` 那条路还缺 `google.api_core`。
所以这一格只做到"结构正确、能 import、缺配置时给出明确提示"，不产生搜索结果。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】向上找含 common/ 的目录；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, describe_adk, short_error, retry_call
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools.vertex_ai_search_tool import VertexAiSearchTool
from google.genai import types

import os

# datastore_id 的格式是完整的资源名：
# projects/{project}/locations/{location}/collections/{collection}/dataStores/{dataStore}
DATASTORE_ID = os.environ.get("DATASTORE_ID")
USER_ID = "user_123"

# ★ 本章新增（ADK 2.13 形态）：书里的 agents.VSearchAgent(...) 已被移除，
#   现在是「普通 LlmAgent + VertexAiSearchTool」，搜索能力退回到 tools=[] 里 ——
#   和第 5 章的主线一致：工具化 = 把特殊能力统一成一个可挂载的对象。
vsearch_agent = LlmAgent(
    name="q2_strategy_vsearch_agent",
    description="Answers questions about Q2 strategy documents using Vertex AI Search.",
    model=adk_model(),                        # 【适配】书里 gemini-2.0-flash-exp 已下线
    tools=[VertexAiSearchTool(data_store_id=DATASTORE_ID)] if DATASTORE_ID else [],
    instruction="Answer using the Vertex AI Search results. Cite where you can.",
)


def call_vsearch_agent(query: str) -> None:
    runner = InMemoryRunner(vsearch_agent)
    session_id = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id=USER_ID, session_id=session_id
    )
    print(f"\nUser: {query}")
    try:
        for event in retry_call(lambda: list(runner.run(
            user_id=USER_ID, session_id=session_id,
            new_message=types.Content(role="user", parts=[types.Part(text=query)]),
        )), label="检索 "):
            if not event.is_final_response() or not event.content:
                continue
            text = getattr(event.content, "text", None) or "".join(
                p.text for p in (event.content.parts or []) if getattr(p, "text", None))
            print("Agent:", (text or "").strip())
            gm = getattr(event, "grounding_metadata", None)
            if gm is not None:
                attrs = getattr(gm, "grounding_attributions", None) or []
                print(f"  (引用来源: {len(attrs)} 个)")
            else:
                print("  (没有 grounding metadata —— 没接 datastore 时属正常)")
    except Exception as e:
        print(f"  ❌ {type(e).__name__}: {short_error(e)}")


if not DATASTORE_ID:
    print("⚠️ 没有 DATASTORE_ID，所以这个 agent 没有挂搜索工具，只能凭模型自身知识回答。")
    print("   要真跑需要：Vertex AI 项目 + datastore，并 export DATASTORE_ID=projects/.../dataStores/...")
    print("   这也是书里这一节在本机无法复现的原因 —— 它依赖的不是 API key，而是一整套云端资源。")
else:
    call_vsearch_agent("Summarize the main points about the Q2 strategy document.")

⚠️ 没有 DATASTORE_ID，所以这个 agent 没有挂搜索工具，只能凭模型自身知识回答。
   要真跑需要：Vertex AI 项目 + datastore，并 export DATASTORE_ID=projects/.../dataStores/...
   这也是书里这一节在本机无法复现的原因 —— 它依赖的不是 API key，而是一整套云端资源。
